# GANs Based City Layout

## Step 0: Downloading Libraries

In [ ]:
!pip install geopandas osmnx tensorflow matplotlib numpy contextily opencv-python

## Step 1: Importing Libraries

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import osmnx as ox
import contextily as ctx
from tensorflow.keras import layers
from pathlib import Path

## Step 2: GPU

In [ ]:
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

In [ ]:
policy = tf.keras.mixed_precision.Policy('mixed_float16')
tf.keras.mixed_precision.set_global_policy(policy)

## Step 3: Download and Save Map Tiles

In [ ]:
def fetch_and_save_tiles(place_name='Gurugram, India', tile_size=256, zoom=16,
                         num_tiles=5000, output_dir='./map_tiles'):
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    boundary = ox.geocode_to_gdf(place_name).to_crs(epsg=3857)
    minx, miny, maxx, maxy = boundary.total_bounds
    lat_step = (maxy - miny) / np.sqrt(num_tiles)
    lon_step = (maxx - minx) / np.sqrt(num_tiles)

    count = 0
    for y in np.arange(miny, maxy, lat_step):
        for x in np.arange(minx, maxx, lon_step):
            if count >= num_tiles:
                break
            try:
                west, south, east, north = x, y, x + lon_step, y + lat_step
                img, _ = ctx.bounds2img(west, south, east, north, zoom=zoom,
                                       source=ctx.providers.OpenStreetMap.Mapnik)
                img = tf.image.resize(img, [tile_size, tile_size]).numpy()
                img_path = os.path.join(output_dir, f"tile_{count}.png")
                cv2.imwrite(img_path, cv2.cvtColor((img * 255).astype(np.uint8),
                                                   cv2.COLOR_RGB2BGR))
                count += 1
            except Exception as e:
                print(f"Error at x={x}, y={y}: {str(e)}")
                continue
        if count >= num_tiles:
            break
    print(f"Saved {count} tiles to {output_dir}")

In [ ]:
# fetch_and_save_tiles()

In [ ]:
# from google.colab import files

# output_dir = './map_tiles'
# zip_file = 'Gurugram_India_map_tiles.zip'
# !zip -r {zip_file} {output_dir}
# files.download(zip_file)
# print(f"Downloaded {zip_file} containing the existing map_tiles folder")

## Creating Dataset

In [ ]:
path = "/content/drive/MyDrive/GANs-Based-City-Layout/Gurugram/map_tiles"

In [ ]:
def create_dataset_from_drive(drive_folder_path=path, tile_size=256, batch_size=16, num_samples=1000):
    if not os.path.exists(drive_folder_path):
        print(f"Error: {drive_folder_path} does not exist. Please check the path.")
        return None

    !ls -l "{drive_folder_path}" | head -n 10
    print(f"Accessing tiles from {drive_folder_path}")

    def load_image(file_path):
        img = tf.io.read_file(file_path)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.2)
        img = img * 2.0 - 1.0
        return img

    dataset = tf.data.Dataset.list_files(f"{drive_folder_path}/*.png")
    dataset = (dataset
               .shuffle(5000)
               .take(num_samples)
               .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
               .batch(batch_size)
               .prefetch(tf.data.AUTOTUNE))
    return dataset

In [ ]:
dataset = create_dataset_from_drive(drive_folder_path=path, num_samples=1000)
if dataset:
    print("Dataset created from Google Drive tiles with augmentation and prefetching.")
else:
    print("Dataset creation failed. Please verify the Drive folder path.")

total 166403
-rw------- 1 root root  35234 Feb 24 14:19 tile_0.png
-rw------- 1 root root  14931 Feb 24 14:25 tile_1000.png
-rw------- 1 root root  12580 Feb 24 14:25 tile_1001.png
-rw------- 1 root root  12456 Feb 24 14:25 tile_1002.png
-rw------- 1 root root  16436 Feb 24 14:25 tile_1003.png
-rw------- 1 root root  25539 Feb 24 14:25 tile_1004.png
-rw------- 1 root root  11286 Feb 24 14:25 tile_1005.png
-rw------- 1 root root   4394 Feb 24 14:25 tile_1006.png
-rw------- 1 root root  17771 Feb 24 14:25 tile_1007.png
Accessing tiles from /content/drive/MyDrive/GANs-Based-City-Layout/Gurugram/map_tiles
Dataset created from Google Drive tiles with augmentation and prefetching.


## Step 5: Making MODEL

In [ ]:
def make_generator(latent_dim=128):
    model = tf.keras.Sequential([
        layers.Input(shape=(latent_dim,)),
        layers.Dense(4 * 4 * 1024),
        layers.LeakyReLU(0.2),
        layers.Reshape((4, 4, 1024)),
        layers.Conv2DTranspose(512, (4, 4), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(256, (4, 4), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(128, (4, 4), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(64, (4, 4), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(32, (4, 4), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(3, (4, 4), strides=(2, 2), padding='same', activation='tanh')
    ])
    return model

In [ ]:
def make_discriminator(img_size=256):
    model = tf.keras.Sequential([
        layers.Input(shape=(img_size, img_size, 3)),
        layers.Conv2D(64, (4, 4), strides=(2, 2), padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Conv2D(128, (4, 4), strides=(2, 2), padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Conv2D(256, (4, 4), strides=(2, 2), padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Conv2D(512, (4, 4), strides=(2, 2), padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(1)
    ])
    return model

In [ ]:
latent_dim = 128
generator = make_generator(latent_dim)
discriminator = make_discriminator()

## Step 6: Loss Function

In [ ]:
def wasserstein_loss(y_true, y_pred):
    return tf.reduce_mean(y_true * y_pred)

In [ ]:
def gradient_penalty(real_images, fake_images, discriminator):
    real_images = tf.cast(real_images, tf.float16)
    fake_images = tf.cast(fake_images, tf.float16)
    alpha = tf.cast(tf.random.uniform([real_images.shape[0], 1, 1, 1], 0., 1.), tf.float16)

    interpolates = alpha * real_images + (1 - alpha) * fake_images
    with tf.GradientTape() as gp_tape:
        gp_tape.watch(interpolates)
        pred = discriminator(interpolates, training=True)
    grads = gp_tape.gradient(pred, [interpolates])[0]
    norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2, 3]))
    return tf.reduce_mean((norm - 1.) ** 2)

## Step 7: Model Training

In [ ]:
gen_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5, beta_2=0.9)
disc_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5, beta_2=0.9)
gp_weight = tf.cast(2.0, tf.float16)

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([tf.shape(images)[0], latent_dim])
    with tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)
        disc_loss_real = wasserstein_loss(tf.ones_like(real_output), real_output)
        disc_loss_fake = wasserstein_loss(-tf.ones_like(fake_output), fake_output)
        gp = gradient_penalty(images, generated_images, discriminator)
        disc_loss = disc_loss_real + disc_loss_fake + gp_weight * gp  # Now all float16

    disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    disc_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

    with tf.GradientTape() as gen_tape:
        generated_images = generator(noise, training=True)
        fake_output = discriminator(generated_images, training=True)
        gen_loss = wasserstein_loss(tf.ones_like(fake_output), fake_output)

    gen_grads = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gen_optimizer.apply_gradients(zip(gen_grads, generator.trainable_variables))

    return gen_loss, disc_loss

In [ ]:
checkpoint_dir = './checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator=generator, discriminator=discriminator,
                                 gen_optimizer=gen_optimizer, disc_optimizer=disc_optimizer)

In [ ]:
def generate_and_save_images(model, epoch, save_folder='./generated_images'):
    noise = tf.random.normal([16, latent_dim])
    predictions = model(noise, training=False)
    predictions = (predictions + 1) / 2.0
    predictions = tf.cast(predictions, tf.float32)  # Ensure float32 for Matplotlib
    Path(save_folder).mkdir(parents=True, exist_ok=True)
    fig, axs = plt.subplots(4, 4, figsize=(12, 12))
    for i, ax in enumerate(axs.flat):
        ax.imshow(predictions[i])
        ax.axis('off')
    plt.savefig(f"{save_folder}/epoch_{epoch}.png")
    plt.close()

In [ ]:
def save_generator_model(model, filepath='./generator_model.keras', zip_filename='generator_model.zip'):
    model.save(filepath, save_format='keras_v3')
    !zip -r {zip_filename} {filepath}
    from google.colab import files
    files.download(zip_filename)
    print(f"Generator model saved and downloaded as '{zip_filename}'")

In [ ]:
def train(dataset, epochs=50, disc_steps=2):
    for epoch in range(epochs):
        for image_batch in dataset:
            for _ in range(disc_steps):
                gen_loss, disc_loss = train_step(image_batch)

        if (epoch + 1) % 5 == 0:
            generate_and_save_images(generator, epoch + 1)
            checkpoint.save(file_prefix=checkpoint_prefix)
            print(f"Epoch {epoch + 1}, Gen Loss: {gen_loss:.4f}, Disc Loss: {disc_loss:.4f}")

In [ ]:
save_generator_model(generator)

  adding: generator_model.keras (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Generator model saved and downloaded as 'generator_model.zip'


In [ ]:
if dataset:
    train(dataset, epochs=55)
    print("Training complete. High-quality city layout images saved in './generated_images'.")
else:
    print("Training skipped. Please verify the Google Drive folder path.")

Epoch 5, Gen Loss: -255.6250, Disc Loss: -17.0000
Epoch 10, Gen Loss: -140.0000, Disc Loss: -7.5664
Epoch 15, Gen Loss: -304.2500, Disc Loss: -30.5000
Epoch 20, Gen Loss: -109.6250, Disc Loss: -48.0312
Epoch 25, Gen Loss: -198.6250, Disc Loss: -54.5625
Epoch 30, Gen Loss: -125.3125, Disc Loss: -15.9219
Epoch 35, Gen Loss: -169.5000, Disc Loss: -39.0000
Epoch 40, Gen Loss: -158.8750, Disc Loss: -73.9375
Epoch 45, Gen Loss: -132.3750, Disc Loss: -51.0625
Epoch 50, Gen Loss: -208.7500, Disc Loss: -30.1094
Epoch 55, Gen Loss: -50.5938, Disc Loss: -53.0312
Training complete. High-quality city layout images saved in './generated_images'.


In [ ]:
import tensorflow as tf
pathModel = "/content/generator_model.keras"
def generate_new_images(model_path=pathModel, num_images=16, save_folder='./new_images', latent_dim=128):
    loaded_generator = tf.keras.models.load_model(model_path)
    noise = tf.random.normal([num_images, latent_dim])
    predictions = loaded_generator(noise, training=False)
    predictions = (predictions + 1) / 2.0
    predictions = tf.cast(predictions, tf.float32)

    if len(predictions.shape) == 2 and predictions.shape[1] == 196608:  # 256*256*3
        predictions = tf.reshape(predictions, [num_images, 256, 256, 3])
    elif predictions.shape[1:] != [256, 256, 3]:
        raise ValueError(f"Unexpected prediction shape: {predictions.shape}. Expected [num_images, 256, 256, 3]")

    Path(save_folder).mkdir(parents=True, exist_ok=True)
    for i in range(num_images):
        plt.figure(figsize=(5, 5))
        plt.imshow(predictions[i])
        plt.axis('off')
        plt.savefig(os.path.join(save_folder, f"new_image_{i}.png"))
        plt.close()
    print(f"{num_images} new images generated and saved to {save_folder}")

In [ ]:
generate_new_images()

16 new images generated and saved to ./new_images


In [ ]:
from google.colab import files

output_dir = './generated_images'
zip_file = 'generated_images.zip'
!zip -r {zip_file} {output_dir}
files.download(zip_file)
print(f"Downloaded {zip_file} containing the generated_images folder")

  adding: generated_images/ (stored 0%)
  adding: generated_images/epoch_40.png (deflated 1%)
  adding: generated_images/epoch_5.png (deflated 0%)
  adding: generated_images/epoch_25.png (deflated 1%)
  adding: generated_images/epoch_55.png (deflated 0%)
  adding: generated_images/epoch_15.png (deflated 0%)
  adding: generated_images/epoch_35.png (deflated 0%)
  adding: generated_images/epoch_10.png (deflated 0%)
  adding: generated_images/epoch_30.png (deflated 0%)
  adding: generated_images/epoch_45.png (deflated 0%)
  adding: generated_images/epoch_50.png (deflated 0%)
  adding: generated_images/epoch_20.png (deflated 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded generated_images.zip containing the generated_images folder


In [ ]:
from google.colab import files

output_dir = './new_images'
zip_file = 'new_images.zip'
!zip -r {zip_file} {output_dir}
files.download(zip_file)
print(f"Downloaded {zip_file} containing the generated_images folder")

  adding: new_images/ (stored 0%)
  adding: new_images/new_image_4.png (deflated 0%)
  adding: new_images/new_image_0.png (deflated 0%)
  adding: new_images/new_image_11.png (deflated 0%)
  adding: new_images/new_image_5.png (deflated 0%)
  adding: new_images/new_image_14.png (deflated 0%)
  adding: new_images/new_image_12.png (deflated 1%)
  adding: new_images/new_image_3.png (deflated 0%)
  adding: new_images/new_image_13.png (deflated 0%)
  adding: new_images/new_image_15.png (deflated 0%)
  adding: new_images/new_image_6.png (deflated 0%)
  adding: new_images/new_image_7.png (deflated 0%)
  adding: new_images/new_image_2.png (deflated 0%)
  adding: new_images/new_image_8.png (deflated 1%)
  adding: new_images/new_image_10.png (deflated 0%)
  adding: new_images/new_image_9.png (deflated 0%)
  adding: new_images/new_image_1.png (deflated 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded new_images.zip containing the generated_images folder
